In [8]:
import pandas as pd
from sklearn.decomposition import PCA

In [9]:
df = pd.read_csv('../data/cleaned_philly_restaurants.csv')

In [10]:
# 2. Clean data (Data Processing / Separation of Concerns)
# Handle missing values and convert to lowercase
df['categories'] = df['categories'].fillna('').str.lower()
# Click once to convert comma-separated text into a 0/1 matrix!
features_matrix = df['categories'].str.get_dummies(sep=', ')

In [11]:
print(f"The dimension of the pure feature table after cleaning：{features_matrix.shape}")

The dimension of the pure feature table after cleaning：(6939, 456)


In [12]:
features_matrix.to_csv('../data/final_features_matrix.csv', index=False)

In [30]:
pca = PCA(n_components=4)
pca.fit(features_matrix)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",4
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD 

In [31]:
# Print out the three most important labels for each principal component
feature_names = features_matrix.columns
for i, component in enumerate(pca.components_):
    top_indices = component.argsort()[-3:][::-1]
    print(f"\nThe core label of the principal component {i+1}:")
    for idx in top_indices:
        print(f"  - {feature_names[idx]} (weight: {component[idx]:.4f})")


The core label of the principal component 1:
  - food (weight: 0.6726)
  - coffee & tea (weight: 0.2503)
  - specialty food (weight: 0.1213)

The core label of the principal component 2:
  - nightlife (weight: 0.5610)
  - bars (weight: 0.5469)
  - food (weight: 0.3696)

The core label of the principal component 3:
  - sandwiches (weight: 0.6101)
  - restaurants (weight: 0.3950)
  - breakfast & brunch (weight: 0.3608)

The core label of the principal component 4:
  - pizza (weight: 0.6220)
  - sandwiches (weight: 0.3395)
  - italian (weight: 0.3259)


### PCA Insights: The 4 Core Business Models (K=4)
Based on our PCA, we will extract exactly 4 components to use as features for the downstream K-Means clustering. Here is the business meaning of each component:

- Component 1: **Light Dining & Cafe Culture** (Driven by: food, coffee & tea)

- Component 2: **Nightlife & Bar Scene** (Driven by: nightlife, bars)

- Component 3: **American Casual & Brunch** (Driven by: sandwiches, breakfast & brunch)

- Component 4: **Italian Pizza & Fast Casual** (Driven by: pizza, italian)

**Why stop at 4?** Components 5 and 6 start blending unrelated features (e.g., mixing coffee with pizza), which creates noise. Keeping exactly 4 gives K-Means the cleanest, most distinct features to cluster commercial zones effectively.

In [ ]:
# Using the previously well-performing PCA model, perform dimensionality reduction on the 456-column features_matrix
# This step instantly reduced the huge matrix of 6939x456 to a streamlined matrix of 6939x4.
pca_scores = pca.transform(features_matrix)

# Convert it into a neat Pandas DataFrame with descriptive column names
df_pca = pd.DataFrame(pca_scores, columns=[
    'PC1_Cafe_Score', 
    'PC2_Nightlife_Score', 
    'PC3_Brunch_Score', 
    'PC4_Pizza_Score'
])

print(df_pca.head())

   PC1_Cafe_Score  PC2_Nightlife_Score  PC3_Brunch_Score  PC4_Pizza_Score
0        0.771661             0.117243          0.377969        -0.389575
1       -0.309880            -0.454242         -0.304166        -0.293109
2       -0.280693            -0.408447         -0.218230        -0.168516
3       -0.998289             0.766304         -0.215518         0.347449
4       -0.374986            -0.592435         -0.040746         0.537202
